# Kak Uli: Chatbot Rekomendasi Cafe & Kuliner Mahasiswa (Groq API)
## Tema: Asisten "Senior Kampus" buat cari tempat nugas & makan yang ramah kantong mahasiswa

Notebook ini membangun chatbot bernama **Kak Uli** yang berperan sebagai senior kampus yang santai dan paham seluk-beluk cafe nugas-able & tempat makan worth it di sekitar kampus. Sekarang chat bot sudah disuntik/ditambahkan data cafe asli surabaya (surabaya_cafes.json), bisa jawab bertahap (streaming), inget riwayat chat, dan punya beberapa fitur tambahan.

Struktur notebook:
1. Instalasi library
2. Menyimpan & memuat API key dengan aman
3. Membuat client API
4. Memuat dataset cafe Surabaya (surabaya_cafes.json)
5. Fungsi filter & Pencarian dataset cafe
6. System prompt dinamis & inisialisasi *conversation history*
7. Fungsi pengiriman pesan dengan **streaming response** + penanganan error
8. Menyimpan & memuat riwayat percakapan (JSON)
9. Statistik percakapan
10. Fitur tambahan preferensi (tematik) & kontrol parameter (mode serius/asik, begadang, terserah, rangkum)
11. Loop chatbot interaktif dengan command khusus (`exit`, `clear`, `budget`, `area`)


In [ ]:
pip install groq # Install library groq agar bisa digunakan

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


## 2. Menyimpan & Memuat API Key

**Jangan pernah menulis API key langsung di dalam kode** (hardcode). 

In [1]:
# 2. MEMUAT API KEY 

import os
from dotenv import load_dotenv

#Coba muat dari file .env (jika ada) dan ambil nilainya
load_dotenv()
api_key = os.environ.get("GROQ_API_KEY")

# Kalau API tidak berjalan / belum diset -> minta input manual (tersembunyi)
if not api_key:
    from getpass import getpass
    api_key = getpass("File .env tidak ditemukan/kosong. Masukkan GROQ_API_KEY kamu: ")

# Validasi akhir
if not api_key:
    raise ValueError("GROQ_API_KEY tidak boleh kosong!.")

# Simpan ke environment agar pustaka Groq bisa membacanya otomatis
os.environ["GROQ_API_KEY"] = api_key
print("API key siap digunakan.")

API key siap digunakan.


## 3. Membuat Client Groq



In [2]:
# 3. MEMBUAT CLIENT

from groq import Groq

client = Groq(api_key=api_key)
print("Groq client berhasil dibuat.")

Groq client berhasil dibuat.


## 4. Memuat Dataset Cafe Surabaya

Supaya Kak Uli nggak ngarang tempat (halusinasi), kita tambahkan data cafe asli dari `surabaya_cafes.json` (misalnya dari repo **surabaya-cafe-api**, berisi ratusan cafe di Surabaya lengkap dengan area, harga, jam buka, rating GMaps, dsb).

Simpan file data nya di folder khusus. Kalau file belum ada, chatbot tetap jalan, tapi Kak Uli akan jujur bilang belum ada data lokal dan tidak akan mengarang nama tempat.


In [3]:
# 4. MEMUAT DATASET CAFE SURABAYA

import json

DATASET_PATH = "../data/surabaya_cafes.json"

def muat_dataset(path=DATASET_PATH):
    """Membaca surabaya_cafes.json dan hanya menyimpan cafe yang statusnya masih aktif di GMaps."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"File '{path}' tidak ditemukan. Kak Uli akan tetap jalan, tapi tanpa data cafe asli.")
        return []
    except json.JSONDecodeError as e:
        print(f"File '{path}' gagal dibaca (format JSON error): {e}")
        return []

    data_aktif = [c for c in data if c.get("gmaps_status", "").lower() == "operational"]
    print(f"Dataset dimuat: {len(data_aktif)} cafe aktif dari total {len(data)} entri.")
    return data_aktif

CAFE_DATA = muat_dataset()

Dataset dimuat: 870 cafe aktif dari total 874 entri.


## 5. Fungsi Filter & Pencarian Dataset Cafe

Fungsi-fungsi ini menyaring `CAFE_DATA` berdasarkan area, budget, jam operasional, dan fasilitas, lalu mengubah hasilnya jadi teks ringkas yang siap disisipkan ke system prompt. Dengan begini, Kak Uli hanya boleh merekomendasikan tempat yang benar-benar ada di dataset.


In [4]:
# 5. FUNGSI FILTER & PENCARIAN DATASET CAFE

import re

def _parse_harga_max(price_range):
    """Ambil angka harga TERTINGGI dari string semisal 'Rp 50-150 rb' -> 150000."""
    if not price_range:
        return None
    angka = re.findall(r"\d+", price_range.replace(".", ""))
    if not angka:
        return None
    angka = [int(a) for a in angka]
    # Asumsikan satuan ribu ('rb') kecuali disebutkan 'jt' (juta)
    faktor = 1_000_000 if "jt" in price_range.lower() else 1_000
    return max(angka) * faktor

def cari_cafe(data, area=None, budget=None, hanya_24jam=False, fasilitas=None, keyword=None, limit=5):
    """Filter dataset cafe berdasarkan beberapa kriteria sekaligus, urut dari rating tertinggi."""
    hasil = data

    if area:
        hasil = [c for c in hasil if area.lower() in (c.get("area") or "").lower()]

    if budget:
        hasil_budget = []
        for c in hasil:
            harga_max = _parse_harga_max(c.get("price_range", ""))
            if harga_max is None or harga_max <= budget:
                hasil_budget.append(c)
        hasil = hasil_budget

    if hanya_24jam:
        hasil = [c for c in hasil if "24 jam" in (c.get("opening_hours") or "").lower()]

    if fasilitas:
        hasil = [c for c in hasil if any(fasilitas.lower() in f.lower() for f in c.get("facilities", []))]

    if keyword:
        hasil = [
            c for c in hasil
            if keyword.lower() in (c.get("name") or "").lower()
            or keyword.lower() in (c.get("description") or "").lower()
        ]

    hasil = sorted(hasil, key=lambda c: c.get("gmaps_rating") or 0, reverse=True)
    return hasil[:limit]

def format_cafe_untuk_prompt(daftar_cafe):
    """Ubah list dict cafe jadi teks ringkas yang siap disisipkan ke prompt Groq."""
    if not daftar_cafe:
        return "Tidak ada data cafe yang cocok di dataset lokal."

    baris = []
    for c in daftar_cafe:
        fasilitas = ", ".join(c.get("facilities") or []) or "-"
        baris.append(
            f"- {c['name']} | Area: {c.get('area', '-')} | Harga: {c.get('price_range', '-')} "
            f"| Jam: {c.get('opening_hours', '-')} | Rating: {c.get('gmaps_rating', '-')} "
            f"({c.get('gmaps_review_count', 0)} ulasan) | Fasilitas: {fasilitas} | Link: {c.get('gmaps_url', '-')}"
        )
    return "\n".join(baris)

## 6. System Prompt Dinamis & Inisialisasi History

Berbeda dari versi sebelumnya yang system prompt-nya statis, sekarang system prompt dibangun ulang setiap giliran lewat `bangun_system_prompt()` supaya selalu berisi **data cafe hasil filter terbaru** sesuai budget, area, dan mode yang sedang aktif (`state`). `messages` (conversation history) sendiri hanya menyimpan pesan `user` & `assistant`. Jadi system prompt disisipkan terpisah tiap kali kirim ke API, biar datanya selalu up to date tanpa membengkakkan riwayat yang disimpan.

In [5]:
# 6. SYSTEM PROMPT DINAMIS & STATE PERCAKAPAN

state = {
    "budget": None,        # contoh: 30000
    "area": None,          # contoh: "Surabaya Timur"
    "mode_24jam": False,   # aktif lewat command 'begadang'
    "temperature": 0.7,
    "max_tokens": 600,
}

SYSTEM_PROMPT_TEMPLATE = """Kamu adalah Kak Uli, senior kampus yang ramah dan santai, ngobrol kayak sama adik tingkat.

Fokus kamu HANYA membantu mahasiswa mencari:
- Cafe yang nugas-able (wifi kenceng, ada colokan, suasana nyaman buat lama-lama)
- Tempat makan yang worth it dan ramah kantong mahasiswa

Aturan menjawab:
- Gunakan Bahasa Indonesia santai, boleh pakai sapaan kayak 'bro/sis' secukupnya, jangan kaku/formal.
- JANGAN PERNAH merekomendasikan tempat yang tidak ada di DATA CAFE TERSEDIA di bawah. Kalau datanya kosong/tidak cocok, jujur bilang belum nemu yang pas, jangan mengarang nama tempat.
- Sebutkan harga, jam buka, rating GMaps, dan link-nya kalau ada di data.
- Kalau mahasiswa belum kasih tau budget atau area, tanya dulu sebelum kasih rekomendasi.
- SELALU tutup rekomendasi dengan pengingat singkat kayak 'cek dulu ya di GMaps/medsos, siapa tau jam buka atau harganya udah berubah'.
- Kalau ditanya di luar topik cafe/tempat makan mahasiswa, arahkan balik dengan santai ke topik utama.

Konteks yang kamu ingat dari mahasiswa ini:
- Budget: {budget}
- Area acuan: {area}
- Mode begadang (cuma cafe 24 jam): {mode_24jam}

DATA CAFE TERSEDIA (hasil filter otomatis dari dataset lokal, JANGAN keluar dari daftar ini):
{data_cafe}
"""

def bangun_system_prompt():
    """Menyusun system prompt terbaru: filter dataset sesuai state, lalu sisipkan ke template."""
    hasil = cari_cafe(
        CAFE_DATA,
        area=state["area"],
        budget=state["budget"],
        hanya_24jam=state["mode_24jam"],
    )
    # Kalau hasil ketat kosong, coba lebih longgar (tanpa budget) biar tetap ada opsi
    if not hasil and (state["area"] or state["budget"]):
        hasil = cari_cafe(CAFE_DATA, area=state["area"], hanya_24jam=state["mode_24jam"])

    return SYSTEM_PROMPT_TEMPLATE.format(
        budget=f"Rp {state['budget']:,}".replace(",", ".") if state["budget"] else "belum diketahui",
        area=state["area"] or "belum diketahui",
        mode_24jam="Aktif" if state["mode_24jam"] else "Tidak aktif",
        data_cafe=format_cafe_untuk_prompt(hasil),
    )

def reset_history():
    """Mengembalikan conversation history ke kondisi kosong (system prompt dikirim terpisah tiap giliran)."""
    return []

messages = reset_history()
print("History percakapan diinisialisasi.")

History percakapan diinisialisasi.


## 7. Fungsi Pengiriman Pesan + Penanganan Error

Model yang dipakai: **`qwen/qwen3.8-27b`** (Qwen, Alibaba Cloud), tersedia gratis di Groq. Statusnya masih *Preview* per dokumentasi Groq (cek [console.groq.com/docs/models](https://console.groq.com/docs/models) kalau suatu saat modelnya berubah/hilang, tinggal ganti `MODEL_NAME`).

Fungsi `kirim_pesan()` mengirim seluruh `messages` (riwayat percakapan) ke API. Kalau terjadi error (API gagal merespons, koneksi putus, dll.), fungsi mengembalikan `None` supaya program **tidak crash** dan history tidak ikut rusak. Sekarang terdapat tambahan `stream=True`: jawaban Kak Uli muncul kata per kata secara real-time, bukan nunggu sekaligus jadi baru muncul. `temperature` dan `max_tokens` juga diambil dari `state`, jadi bisa diubah lewat command (`mode serius`, `suhu`, `panjang`) di bagian 10.


In [6]:
# 7. FUNGSI PENGIRIMAN PESAN (STREAMING)

MODEL_NAME = "qwen/qwen3.8-27b"

def kirim_pesan_streaming(messages_lengkap):
    """
    Mengirim system prompt + riwayat percakapan ke Groq API dengan stream=True,
    lalu mencetak jawaban Kak Uli bertahap (token per token) ke layar.
    Mengembalikan jawaban lengkap (string) atau None kalau terjadi error.
    """
    try:
        stream = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages_lengkap,
            temperature=state["temperature"],
            max_tokens=state["max_tokens"],
            stream=True,
        )
    except Exception as e:
        print("\n Terjadi error saat memanggil API:")
        print(e)
        return None

    print("\nKak Uli : ", end="", flush=True)
    jawaban_lengkap = ""
    try:
        for chunk in stream:
            potongan = chunk.choices[0].delta.content or ""
            print(potongan, end="", flush=True)
            jawaban_lengkap += potongan
    except Exception as e:
        print(f"\n[Streaming terputus di tengah jalan: {e}]")

    print("\n")
    return jawaban_lengkap if jawaban_lengkap else None

## 8. Menyimpan & Memuat Riwayat Percakapan

Riwayat (`messages`, tanpa system prompt) disimpan ke file JSON di folder `riwayat_chat/`, jadi mahasiswa bisa lanjut chat besok tanpa mulai dari nol. Format JSON dipilih karena strukturnya sama persis dengan `list of dict` (`role` & `content`) yang memang dipakai Groq API.

In [7]:
# 8. MENYIMPAN & MEMUAT RIWAYAT PERCAKAPAN

from datetime import datetime

FOLDER_RIWAYAT = "riwayat_chat"
os.makedirs(FOLDER_RIWAYAT, exist_ok=True)

def simpan_riwayat(messages_sesi, nama_file=None):
    """Simpan riwayat percakapan (list of dict) ke file JSON di folder riwayat_chat/."""
    if not messages_sesi:
        print("Belum ada percakapan yang bisa disimpan.\n")
        return None

    if nama_file is None:
        nama_file = f"riwayat_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    if not nama_file.endswith(".json"):
        nama_file += ".json"
    path = os.path.join(FOLDER_RIWAYAT, nama_file)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(messages_sesi, f, ensure_ascii=False, indent=2)

    print(f"Riwayat berhasil disimpan ke '{path}'\n")
    return path

def muat_riwayat(nama_file):
    """Muat kembali riwayat percakapan dari file JSON di folder riwayat_chat/."""
    if not nama_file.endswith(".json"):
        nama_file += ".json"
    path = os.path.join(FOLDER_RIWAYAT, nama_file)
    if not os.path.exists(path):
        print(f"File '{path}' tidak ditemukan.\n")
        return None

    with open(path, "r", encoding="utf-8") as f:
        riwayat = json.load(f)

    print(f"Riwayat '{path}' berhasil dimuat ({len(riwayat)} pesan).\n")
    return riwayat

def daftar_file_riwayat():
    """Menampilkan semua file riwayat yang tersimpan di folder riwayat_chat/."""
    file_list = sorted(f for f in os.listdir(FOLDER_RIWAYAT) if f.endswith(".json"))
    if not file_list:
        print("Belum ada riwayat yang tersimpan.\n")
    else:
        print("Daftar riwayat tersimpan:")
        for f in file_list:
            print(f"  - {f}")
        print()

## 9. Statistik Percakapan

Sebagai anak Sains Data Terapan, kita tambahkan analitik kecil-kecilan: total pesan, kata kunci yang paling sering muncul dari pertanyaan mahasiswa, dan area mana yang paling sering ditanyain (dicocokkan dengan kolom `area` di `CAFE_DATA`).

In [8]:
# 9. STATISTIK PERCAKAPAN

from collections import Counter

STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "ini", "itu", "kak",
    "uli", "aku", "kamu", "ya", "nih", "dong", "sih", "gak", "ga", "nggak",
    "banget", "apa", "atau", "ada", "juga", "saya", "kita", "mau", "bisa",
    "kalo", "kalau", "buat", "gimana", "cari", "mana",
}

def hitung_statistik(messages_sesi):
    """Menampilkan ringkasan statistik dari percakapan yang sedang berjalan."""
    pesan_user = [m["content"] for m in messages_sesi if m["role"] == "user"]
    pesan_asisten = [m["content"] for m in messages_sesi if m["role"] == "assistant"]

    semua_kata = []
    for teks in pesan_user:
        kata = re.findall(r"[a-zA-Z]+", teks.lower())
        semua_kata.extend(k for k in kata if k not in STOPWORDS and len(k) > 2)
    kata_terpopuler = Counter(semua_kata).most_common(5)

    daftar_area = {c.get("area") for c in CAFE_DATA if c.get("area")}
    penyebutan_area = Counter()
    for teks in pesan_user:
        for area in daftar_area:
            if area.lower() in teks.lower():
                penyebutan_area[area] += 1

    print("=" * 50)
    print("STATISTIK PERCAKAPAN")
    print("=" * 50)
    print(f"Total pesan kamu      : {len(pesan_user)}")
    print(f"Total balasan Kak Uli : {len(pesan_asisten)}")
    print(f"Budget yang diketahui : {state['budget'] if state['budget'] else '-'}")
    print(f"Area acuan saat ini   : {state['area'] if state['area'] else '-'}")
    print(f"Mode begadang         : {'Aktif' if state['mode_24jam'] else 'Tidak aktif'}")
    if kata_terpopuler:
        print("Kata kunci paling sering kamu sebut:")
        for kata, jumlah in kata_terpopuler:
            print(f"  - {kata} ({jumlah}x)")
    if penyebutan_area:
        print("Area paling sering ditanyakan:")
        for area, jumlah in penyebutan_area.most_common(3):
            print(f"  - {area} ({jumlah}x)")
    print("=" * 50 + "\n")


## 10. Fitur Tambahan Tematik & Kontrol Parameter

- `mode serius` / `mode asik` -> ubah `temperature` biar jawaban lebih fokus atau lebih kreatif
- `suhu <0-1>` -> atur `temperature` manual
- `panjang <pendek/sedang/panjang>` -> atur estimasi panjang jawaban (`max_tokens`)
- `begadang` -> toggle mode cuma rekomendasi cafe 24 jam
- `terserah` -> "gacha", Kak Uli pilihin 1 cafe random sesuai budget/area yang lagi aktif
- `rangkum` -> tarik semua nama cafe yang kebahas di sesi ini, simpan ke `daftar_nongkrong.txt`

In [9]:
# 10. FITUR TAMBAHAN TEMATIK & KONTROL PARAMETER

import random

PANJANG_MAP = {"pendek": 200, "sedang": 600, "panjang": 1000}

def atur_mode_parameter(perintah):
    """Command 'mode serius' / 'mode asik' untuk mengubah temperature secara instan."""
    if perintah == "mode serius":
        state["temperature"] = 0.2
        print("Mode serius aktif. Kak Uli bakal jawab lebih fokus & to the point.\n")
    elif perintah == "mode asik":
        state["temperature"] = 0.9
        print("Mode asik aktif. Kak Uli bakal jawab lebih santai & kreatif.\n")

def fitur_terserah():
    """Gacha: pilih 1 cafe secara acak, mengikuti budget/area/mode yang lagi aktif."""
    kandidat = cari_cafe(
        CAFE_DATA, area=state["area"], budget=state["budget"],
        hanya_24jam=state["mode_24jam"], limit=50,
    )
    if not kandidat:
        kandidat = CAFE_DATA
    if not kandidat:
        print("Kak Uli : Waduh, dataset-nya kosong nih, Kak Uli nggak bisa gacha.\n")
        return None
    return random.choice(kandidat)

def fitur_rangkum(messages_sesi):
    """Ekstrak nama cafe dari dataset yang kesebut sepanjang chat, simpan ke file .txt."""
    teks_gabungan = " ".join(m["content"] for m in messages_sesi)
    ditemukan = []
    for c in CAFE_DATA:
        if c["name"].lower() in teks_gabungan.lower() and c["name"] not in ditemukan:
            ditemukan.append(c["name"])

    if not ditemukan:
        print("Kak Uli : Belum ada nama cafe spesifik yang kebahas nih, ngobrol dulu yuk!\n")
        return

    with open("daftar_nongkrong.txt", "w", encoding="utf-8") as f:
        f.write("Daftar tempat nongkrong hasil rangkuman chat sama Kak Uli:\n\n")
        for i, nama in enumerate(ditemukan, 1):
            f.write(f"{i}. {nama}\n")

    print(f"{len(ditemukan)} tempat berhasil dirangkum ke 'daftar_nongkrong.txt'\n")


## 11. Loop Chatbot Interaktif

Command yang tersedia sekarang:

| Command | Fungsi |
|---|---|
| `exit` | keluar dari chatbot |
| `clear` | hapus riwayat percakapan sesi ini |
| `help` | tampilkan menu command lagi |
| `budget <angka>` | set budget, contoh: `budget 30000` |
| `area <lokasi>` | set area acuan, contoh: `area Surabaya Timur` |
| `begadang` | toggle mode cuma cafe 24 jam |
| `terserah` | gacha 1 cafe random sesuai budget/area kamu |
| `rangkum` | simpan cafe yang dibahas ke `daftar_nongkrong.txt` |
| `stats` | lihat statistik percakapan |
| `mode serius` / `mode asik` | atur gaya jawaban Kak Uli |
| `suhu <0-1>` | atur `temperature` manual |
| `panjang <pendek/sedang/panjang>` | atur estimasi panjang jawaban |
| `simpan [nama_file]` | simpan riwayat chat ke JSON |
| `muat <nama_file>` | muat ulang riwayat chat |
| `daftar riwayat` | lihat semua file riwayat tersimpan |

Semua pesan biasa (di luar command) tetap dikirim ke Groq dengan **streaming response**, dan system prompt-nya selalu disuntik ulang dengan data cafe terbaru sesuai `state` yang aktif.


In [10]:
# 11. CHATBOT LOOP

def tampilkan_bantuan():
    print("=" * 60)
    print("KAK ULI — REKOMENDASI CAFE & KULINER MAHASISWA SURABAYA")
    print("=" * 60)
    print("Ketik 'exit'                            -> keluar")
    print("Ketik 'clear'                           -> hapus riwayat percakapan sesi ini")
    print("Ketik 'help'                            -> tampilkan menu ini lagi")
    print("Ketik 'budget <angka>'                  -> set budget, contoh: budget 30000")
    print("Ketik 'area <lokasi>'                   -> set area acuan, contoh: area Surabaya Timur")
    print("Ketik 'begadang'                        -> toggle mode cuma cafe 24 jam")
    print("Ketik 'terserah'                        -> gacha 1 cafe random")
    print("Ketik 'rangkum'                         -> simpan cafe yang dibahas ke .txt")
    print("Ketik 'stats'                           -> lihat statistik percakapan")
    print("Ketik 'mode serius' / 'mode asik'       -> atur gaya jawaban Kak Uli")
    print("Ketik 'suhu <0-1>'                      -> atur temperature manual")
    print("Ketik 'panjang <pendek/sedang/panjang>' -> atur estimasi panjang jawaban")
    print("Ketik 'simpan [nama_file]'              -> simpan riwayat chat ke JSON")
    print("Ketik 'muat <nama_file>'                -> muat ulang riwayat chat")
    print("Ketik 'daftar riwayat'                  -> lihat semua file riwayat tersimpan")
    print("=" * 60 + "\n")

tampilkan_bantuan()
messages = reset_history()

while True:

    user_input = input("Kamu : ").strip()

    if not user_input:
        print("Silakan ketik pesan dulu ya.\n")
        continue

    perintah = user_input.lower()

    # ---- exit ----
    if perintah == "exit":
        print("\nKak Uli : Oke, semoga nugasnya lancar! Sampai jumpa \n")
        break

    # ---- clear ----
    if perintah == "clear":
        messages = reset_history()
        print("\nRiwayat percakapan telah dihapus. Mulai obrolan baru.\n")
        continue

    # ---- help ----
    if perintah == "help":
        tampilkan_bantuan()
        continue

    # ---- budget ----
    if perintah.startswith("budget "):
        try:
            state["budget"] = int(re.sub(r"[^\d]", "", perintah[len("budget "):]))
            print(f"(Kamu bilang ke Kak Uli: budget kamu sekitar Rp{state['budget']})\n")
        except ValueError:
            print("Format salah. Contoh: budget 30000\n")
        continue

    # ---- area ----
    if perintah.startswith("area "):
        state["area"] = user_input[len("area "):].strip()
        print(f"(Kamu bilang ke Kak Uli: area acuan kamu {state['area']})\n")
        continue

    # ---- begadang ----
    if perintah == "begadang":
        state["mode_24jam"] = not state["mode_24jam"]
        status = "aktif" if state["mode_24jam"] else "nonaktif"
        print(f"Mode begadang sekarang {status}.\n")
        continue

    # ---- mode serius / asik ----
    if perintah in ("mode serius", "mode asik"):
        atur_mode_parameter(perintah)
        continue

    # ---- suhu ----
    if perintah.startswith("suhu "):
        try:
            nilai = float(perintah[len("suhu "):])
            state["temperature"] = max(0.0, min(1.0, nilai))
            print(f"Temperature diatur ke {state['temperature']}\n")
        except ValueError:
            print("Format salah. Contoh: suhu 0.5\n")
        continue

    # ---- panjang ----
    if perintah.startswith("panjang "):
        pilihan = perintah[len("panjang "):].strip()
        if pilihan in PANJANG_MAP:
            state["max_tokens"] = PANJANG_MAP[pilihan]
            print(f"Panjang jawaban diatur ke '{pilihan}' (~{state['max_tokens']} token)\n")
        else:
            print("Pilihan: pendek / sedang / panjang\n")
        continue

    # ---- terserah (gacha) ----
    if perintah == "terserah":
        pilihan = fitur_terserah()
        if pilihan:
            prompt_tersembunyi = (
                f"Aku bingung mau kemana, tolong pilihkan satu tempat buat aku: {pilihan['name']}. "
                "Jelaskan kenapa tempat ini asik buat dicoba."
            )
            messages.append({"role": "user", "content": prompt_tersembunyi})
            system_prompt = bangun_system_prompt()
            jawaban = kirim_pesan_streaming([{"role": "system", "content": system_prompt}] + messages)
            if jawaban is not None:
                messages.append({"role": "assistant", "content": jawaban})
            else:
                messages.pop()
        continue

    # ---- rangkum ----
    if perintah == "rangkum":
        fitur_rangkum(messages)
        continue

    # ---- stats ----
    if perintah == "stats":
        hitung_statistik(messages)
        continue

    # ---- simpan ----
    if perintah.startswith("simpan"):
        bagian = user_input.split(" ", 1)
        nama_file = bagian[1].strip() if len(bagian) > 1 else None
        simpan_riwayat(messages, nama_file)
        continue

    # ---- muat ----
    if perintah.startswith("muat "):
        nama_file = user_input[len("muat "):].strip()
        hasil_muat = muat_riwayat(nama_file)
        if hasil_muat is not None:
            messages = hasil_muat
            print("Riwayat berhasil dimuat, lanjut ngobrol yuk!\n")
        continue

    # ---- daftar riwayat ----
    if perintah == "daftar riwayat":
        daftar_file_riwayat()
        continue

    # ---- pesan biasa -> kirim ke Groq dengan streaming ----
    print(f"Kamu : {user_input}")
    messages.append({"role": "user", "content": user_input})

    system_prompt = bangun_system_prompt()
    jawaban = kirim_pesan_streaming([{"role": "system", "content": system_prompt}] + messages)

    if jawaban is not None:
        messages.append({"role": "assistant", "content": jawaban})
    else:
        messages.pop()


KAK ULI — REKOMENDASI CAFE & KULINER MAHASISWA SURABAYA
Ketik 'exit'                            -> keluar
Ketik 'clear'                           -> hapus riwayat percakapan sesi ini
Ketik 'help'                            -> tampilkan menu ini lagi
Ketik 'budget <angka>'                  -> set budget, contoh: budget 30000
Ketik 'area <lokasi>'                   -> set area acuan, contoh: area Surabaya Timur
Ketik 'begadang'                        -> toggle mode cuma cafe 24 jam
Ketik 'terserah'                        -> gacha 1 cafe random
Ketik 'rangkum'                         -> simpan cafe yang dibahas ke .txt
Ketik 'stats'                           -> lihat statistik percakapan
Ketik 'mode serius' / 'mode asik'       -> atur gaya jawaban Kak Uli
Ketik 'suhu <0-1>'                      -> atur temperature manual
Ketik 'panjang <pendek/sedang/panjang>' -> atur estimasi panjang jawaban
Ketik 'simpan [nama_file]'              -> simpan riwayat chat ke JSON
Ketik 'muat <nama_file>'  

(Kamu bilang ke Kak Uli: area acuan kamu Surabaya Timur)

(Kamu bilang ke Kak Uli: budget kamu sekitar Rp20000)

Kamu : rekomendasi dong cafe yang enak nugas dan ramah di kantong mahasiswa

Kak Uli : Yo, bro/sis! Siap bantu carikan tempat nugas yang enak dan affordable di Surabaya Timur.

Berdasarkan data yang ada, ini beberapa opsi yang bisa kamu pertimbangkan:

1.  **Angkringan & Cafe Basecamp 17**
    *   **Kenapa cocok:** Karena namanya angkringan + cafe, biasanya harga makanannya lebih masuk akal buat kantong mahasiswa dibanding cafe premium. Ratingnya juga oke (4/5 dari 132 ulasan), jadi kualitasnya udah teruji.
    *   **Jam Buka:** Senin-Sabtu 10:00-21:00.
    *   **Link:** [Cek di Maps](https://www.google.com/maps/place/Harman+Cafe/data=!4m7!3m6!1s0x2dd7fb1f413f0a93:0x3c57e95f0613ab5c!8m2!3d-7.2999956!4d112.7667953!16s%2Fg%2F11q2tjbf1_!19sChIJkwo_QR_71y0RXKsTBl_pVzw)

2.  **Cafe Goede**
    *   **Kenapa cocok:** Ratingnya tinggi banget (4/5 dengan 1080 ulasan!), artinya suasan